In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np

# # Leer el archivo Excel especificando que la columna de cédulas es tipo texto
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    archivo_orig_1 = pd.read_sql("select * from CUN_REPOSITORIO.comisiones.TELECAMPUS_VENTAS_TIEMPOREAL", engine2_1)
    df_archivo = archivo_orig_1.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
#-----------------------------------
#-----------------------------------
df_archivo

Conexion a la base de datos realizada
Conexión cerrada


,DOC_ALUM,COD_UNI,NOM_PROGRAMA,TIPO,producto,NUEVO,EST_MATRICULADO,PROPIETARIO_ZOHO,UNIDADNEGOCIO,vendedor,...,VINCULACIONES,val_x_fuerz,val_x_fuerz_2,Reintegro,Reintegro_Desercion,llave_tarifa_t_2,Reint_Stv,EstadoFinal,NOMBRE,LIQUIDADO
0,98765078,ESP27,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,Postgrado,Postgrado,NUEVO,1-Activo,TELECAMPUS ITAGUI,TELECAMPUS,TELECAMPUS ITAGUI,...,0,0,0,0,None,None,0,VALIDO,CARLOS MARIO OSORIO SUAREZ,0
1,98761441,ESP22,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,Postgrado,Postgrado,ANTIGUO,1-Activo,TELECAMPUS ITAGUI,TELECAMPUS,TELECAMPUS ITAGUI,...,0,0,0,0,None,None,0,VALIDO,MILLER ALBERTO CARTAGENA ALZATE,0
2,98761441,ESP22,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,Postgrado,Postgrado,NUEVO,1-Activo,TELECAMPUS ITAGUI,TELECAMPUS,TELECAMPUS ITAGUI,...,0,0,0,0,None,None,0,VALIDO,MILLER ALBERTO CARTAGENA ALZATE,0
3,98712060,ESP27,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,Postgrado,Postgrado,ANTIGUO,1-Activo,TELECAMPUS ENGATIVA,TELECAMPUS,TELECAMPUS ENGATIVA,...,0,0,0,0,None,None,0,VALIDO,JORGE MARIO TAMAYO HOYOS,0
4,98712060,ESP27,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,Postgrado,Postgrado,NUEVO,1-Activo,TELECAMPUS ENGATIVA,TELECAMPUS,TELECAMPUS ENGATIVA,...,0,0,0,0,None,None,0,VALIDO,JORGE MARIO TAMAYO HOYOS,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3106,1000117461,VTC06,CONTADURIA PUBLICA,Pregrado,Pregrado,NUEVO,1-Activo,TELECAMPUS BOSA,TELECAMPUS,TELECAMPUS BOSA,...,0,0,0,0,None,None,0,VALIDO,CRISTIAN JAIR PARRAGA GARZON,0
3107,10000843,ESP23,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,Postgrado,Postgrado,ANTIGUO,1-Activo,TELECAMPUS ENGATIVA,TELECAMPUS,TELECAMPUS ENGATIVA,...,0,0,0,0,None,None,0,VALIDO,BIBIANO ANTONIO OCAMPO HERNANDEZ,0
3108,10000843,ESP23,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,Postgrado,Postgrado,NUEVO,1-Activo,TELECAMPUS ENGATIVA,TELECAMPUS,TELECAMPUS ENGATIVA,...,0,0,0,0,None,None,0,VALIDO,BIBIANO ANTONIO OCAMPO HERNANDEZ,0
3109,1000031926,VTM62,DIRECCION Y PRODUCCION DE MEDIOS AUDIOVISUALES,Pregrado,Pregrado,NUEVO,1-Activo,TELECAMPUS CHIA CENTRO,TELECAMPUS,TELECAMPUS CHIA CENTRO,...,0,0,0,0,None,None,0,VALIDO,Daniela Yeraldin Romero Romero,0


In [2]:
df_archivo.rename(columns={
    'Comision_Final_N': 'COMISION_NN',
    'Refer': 'REFERIDO',
    'Propietario Diferente':'pert_contac',
    'ORDEN_RECAUDO' : 'orden',
    'Deuda_CT' : 'AVAL_Y_GASTOS_TECNICOS'
    
}, inplace=True)

In [3]:
df_archivo['REFERIDO'] = '0'
df_archivo['NOVEDAD'] = '0'
df_archivo['ORDEN_MAYOR'] = df_archivo['VAL_LIQUIDADO']
df_archivo['VALOR_ORDEN'] = df_archivo['ORDEN_MAYOR']

In [4]:
df_archivo.drop(columns={'EstadoFinal','Base_Rec'}, inplace=True)
df_archivo.shape

(3111, 63)

In [5]:
df_archivo['VALOR_MATRICULA_R'] = df_archivo['VALOR_DEUDA'] 
df_archivo['10 PORCT'] = df_archivo['VALOR_DEUDA'] * 0.10

In [6]:
df_archivo = df_archivo[['DOC_ALUM', 'NOMBRE', 'COD_UNI', 'NOM_PROGRAMA', 'TIPO', 'producto',
       'NUEVO', 'EST_MATRICULADO', 'PROPIETARIO_ZOHO', 'UNIDADNEGOCIO',
       'vendedor', 'PERIODO', 'ESTADO_PAGO', 'FECHA_DE_RECAUDO', 'MES_RECAUDO',
       'orden', 'VAL_LIQUIDADO', 'RECARGO', 'ADICIONALES', 'VAL_BECDTOS',
       'NCR_ANULACION', 'VALOR_DEDUCIBLE_INGLES', 'num_pagos', 'VALOR_RECAUDO',
       'RECAUDO_TOTAL', 'VALOR_MATRICULA', 'VALOR_DEUDA', 'PESO_MATRICULA',
       'PESO_INGLES', 'PESO_SEGURO', 'PC_COMISION_MATRICULA',
       'PC_COMISION_INGLES', 'PERIODO_FINANCIERO', 'BASE_COMISIONABLE',
       'TOTAL_A_COMISIONAR_MATRICULA', 'TOTAL_A_COMISIONAR_INGLES',
       'TOTAL_COMISIONAR', 'PC_PAGADO', 'BASE_MATRICULA', 'BASE_INGLES',
       'COMISION_MATRICULA', 'COMISION_INGLES', 'MES_FACTURAR', 'TOTAL_PAGADO',
       'PC_PAGADO_TELECAMPUS', 'ESTADO_TEL', 'Tiene_Resta_Historica',
       'VALOR_N_RECAUDO', 'ORDEN_MAYOR', 'VALOR_MATRICULA_R', '10 PORCT',
       'REFERIDO', 'pert_contac', 'NOVEDAD', 'val_x_fuerz', 'val_x_fuerz_2',
       'Reintegro', 'Reintegro_Desercion', 'Reint_Stv', 'VINCULACIONES',
       'VALOR_ORDEN', 'AVAL_Y_GASTOS_TECNICOS', 'COMISION_NN', 'LIQUIDADO']]

In [7]:
# ES NECESARIO REEMPLAZAR TODOS LOS VACIOS POR 0
df_archivo.fillna(0, inplace=True)

# DESPUES CONVIERTO TODO A STRING
df_archivo['DOC_ALUM']=df_archivo['DOC_ALUM'].astype(int).astype(str)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_18444\1511698490.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_archivo.fillna(0, inplace=True)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_18444\1511698490.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_archivo.fillna(0, inplace=True)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_18444\1511698490.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

In [8]:
#SIEMPRE CONVERTIR Y MIRAR PERIODO FINANCIERO - no puede estar vacio
df_archivo['PERIODO_FINANCIERO']=df_archivo['PERIODO_FINANCIERO'].astype(int).astype(str)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_18444\983519478.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_archivo['PERIODO_FINANCIERO']=df_archivo['PERIODO_FINANCIERO'].astype(int).astype(str)


In [9]:
df_archivo['orden']=df_archivo['orden'].replace('Null','0')
df_archivo['orden']=df_archivo['orden'].astype(int).astype(str)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_18444\130656013.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_archivo['orden']=df_archivo['orden'].replace('Null','0')
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_18444\130656013.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_archivo['orden']=df_archivo['orden'].astype(int).astype(str)


In [10]:
df_archivo['VALOR_RECAUDO']=df_archivo['VALOR_RECAUDO'].astype(float)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_18444\2855871786.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_archivo['VALOR_RECAUDO']=df_archivo['VALOR_RECAUDO'].astype(float)


In [11]:
df_archivo = df_archivo.drop_duplicates()
df_archivo.shape

(3111, 64)

In [12]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np

# # Leer el archivo Excel especificando que la columna de cédulas es tipo texto
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_novalidos = pd.read_sql("select * from CUN_REPOSITORIO.comisiones.TELECAMPUS_NO_VALIDOS", engine2_1)
    
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_novalidos
#-----------------------------------
#-----------------------------------

Conexion a la base de datos realizada
Conexión cerrada


,DOC_ALUM,COD_UNI,NOM_PROGRAMA,TIPO,producto,NUEVO,EST_MATRICULADO,PROPIETARIO_ZOHO,UNIDADNEGOCIO,vendedor,...,COMISION_MATRICULA,COMISION_INGLES,MES_FACTURAR,TOTAL_PAGADO,PC_PAGADO_TELECAMPUS,Tiene_Resta_Historica,Deuda_CT,VALOR_N_RECAUDO,Comision_Final_N,EstadoFinal
0,80251840,ESP24,ESPECIALIZACIÓN EN ALTA GERENCIA,Postgrado,Postgrado,NUEVO,1-Activo,TELECAMPUS ENGATIVA,TELECAMPUS,TELECAMPUS ENGATIVA,...,,,SEPTIEMBRE,0,0,NO,20000.0,327900.0,0.0,NO VALIDO OTRA FUERZA
1,1193546554,VTC09,INGENIERIA INDUSTRIAL,Pregrado,Pregrado,NUEVO,1-Activo,TELECAMPUS ENGATIVA,TELECAMPUS,TELECAMPUS ENGATIVA,...,0.0,41029.5,SEPTIEMBRE,0,0,NO,20000.0,390295.0,0.0,NO VALIDO OTRA FUERZA
2,1144186941,VT15G,INGENIERIA DE SISTEMAS,Pregrado,Pregrado,NUEVO,1-Activo,TELECAMPUS ZIPAQUIRA,TELECAMPUS,TELECAMPUS ZIPAQUIRA,...,376530.0,20000.0,SEPTIEMBRE,0,0,NO,0.0,977060.0,0.0,NO VALIDO REINTEGRO
3,1110576099,ESP23,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,Postgrado,Postgrado,NUEVO,1-Activo,TELECAMPUS GUAVIO,TELECAMPUS,TELECAMPUS GUAVIO,...,,,SEPTIEMBRE,0,0,NO,20000.0,741632.0,0.0,NO VALIDO VINCULACIONES
4,1085899449,ESP23,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,Postgrado,Postgrado,NUEVO,1-Activo,TELECAMPUS GUAVIO,TELECAMPUS,TELECAMPUS GUAVIO,...,,,SEPTIEMBRE,0,0,NO,20000.0,480000.0,0.0,NO VALIDO OTRA FUERZA
5,1077090634,VTC09,INGENIERIA INDUSTRIAL,Pregrado,Pregrado,NUEVO,1-Activo,TELECAMPUS BOSA,TELECAMPUS,TELECAMPUS BOSA,...,236720.0,20000.0,SEPTIEMBRE,0,0,NO,0.0,697440.0,0.0,NO VALIDO REINTEGRO
6,1075303034,VT15G,INGENIERIA DE SISTEMAS,Pregrado,Pregrado,NUEVO,1-Activo,TELECAMPUS ITAGUI,TELECAMPUS,TELECAMPUS ITAGUI,...,236720.0,20000.0,SEPTIEMBRE,0,0,NO,0.0,697440.0,0.0,NO VALIDO REINTEGRO
7,1075224184,ESP20,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,Postgrado,Postgrado,NUEVO,1-Activo,TELECAMPUS SUBA,TELECAMPUS,TELECAMPUS SUBA,...,,,SEPTIEMBRE,0,0,NO,20000.0,327900.0,0.0,NO VALIDO OTRA FUERZA
8,1065592428,VTE01,ADMINISTRACIÓN DE EMPRESAS,Pregrado,Pregrado,NUEVO,1-Activo,TELECAMPUS SUBA,TELECAMPUS,TELECAMPUS SUBA,...,54445.532117458133,4599.9942647396192,SEPTIEMBRE,0,0,NO,20000.0,140411.0,0.0,NO VALIDO OTRA FUERZA
9,1049626991,VTM62,DIRECCION Y PRODUCCION DE MEDIOS AUDIOVISUALES,Pregrado,Pregrado,NUEVO,1-Activo,TELECAMPUS MEDELLIN,TELECAMPUS,TELECAMPUS MEDELLIN,...,0.0,2300.0,SEPTIEMBRE,0,0,NO,0.0,23000.0,0.0,NO VALIDO REINTEGRO


In [13]:
with pd.ExcelWriter('LIQUIDACION_TELECAMPUS_SEPTIEMBRE_2026_2.xlsx', engine='openpyxl') as writer:
    df_archivo.to_excel(writer, sheet_name='VALIDOS', index=False)
    df_novalidos.to_excel(writer, sheet_name='NO VALIDOS', index=False)

In [14]:
rr
##df_archivo.to_excel('LIQUIDACION_TELECAMPUS_OCTUBRE.xlsx', index=False)

NameError: name 'rr' is not defined

In [ ]:
# ES NECESARIO REEMPLAZAR TODOS LOS VACIOS POR 0
df_archivo.fillna(0, inplace=True)

# DESPUES CONVIERTO TODO A STRING
df_archivo['DOC_ALUM']=df_archivo['DOC_ALUM'].astype(int).astype(str)

In [ ]:
#SIEMPRE CONVERTIR Y MIRAR PERIODO FINANCIERO - no puede estar vacio
df_archivo['PERIODO_FINANCIERO']=df_archivo['PERIODO_FINANCIERO'].astype(int).astype(str)

In [ ]:
df_archivo['orden']=df_archivo['orden'].replace('Null','0')
df_archivo['orden']=df_archivo['orden'].astype(int).astype(str)

In [ ]:
df_archivo['VALOR_RECAUDO']=df_archivo['VALOR_RECAUDO'].astype(float)

In [ ]:
df_archivo = df_archivo.drop_duplicates()
df_archivo.shape

(1557, 64)

In [ ]:
df_archivo['financiado'] = ((df_archivo['ORDEN_MAYOR']+df_archivo['ADICIONALES']+df_archivo['VALOR_DEDUCIBLE_INGLES']+df_archivo['VAL_BECDTOS']) == df_archivo['VALOR_N_RECAUDO']).astype(int)
df_archivo

,DOC_ALUM,NOMBRE,COD_UNI,NOM_PROGRAMA,TIPO,producto,NUEVO,EST_MATRICULADO,PROPIETARIO_ZOHO,UNIDADNEGOCIO,...,val_x_fuerz_2,Reintegro,Reintegro_Desercion,Reint_Stv,VINCULACIONES,VALOR_ORDEN,AVAL_Y_GASTOS_TECNICOS,COMISION_NN,LIQUIDADO,financiado
0,98715440,JORGE ARMANDO OCAMPO,VPC64,ADMINISTRACION DE EMPRESAS AGROINDUSTRIALES,Pregrado,Pregrado,NUEVO,1-Activo,BIZEN,TELECAMPUS,...,0,0,0,0,0,5423000,20000.0,4.276000e+04,0,0
1,98698820,MANUEL ESTEBAN AGUDELO PALACIO,VT15G,INGENIERIA DE SISTEMAS,Pregrado,Pregrado,NUEVO,1-Activo,TUNJA,TELECAMPUS,...,0,0,0,0,0,0,20000.0,4.122257e+05,0,0
2,98671662,ANDRÉS FELIPE BETANCUR LÓPEZ,ESP28,ESPECIALIZACION EN ANALITICA DE DATOS,Postgrado,Postgrado,NUEVO,1-Activo,TUNJA,TELECAMPUS,...,0,0,0,0,0,6598000,0.0,1.001700e+06,0,1
3,9863459,DAVID ANDREY QUICENO GARCIA,ESP24,ESPECIALIZACIÓN EN ALTA GERENCIA,Postgrado,Postgrado,NUEVO,1-Activo,NEIVA,TELECAMPUS,...,0,0,0,0,0,6598000,0.0,1.001700e+06,0,1
4,98578991,FREDY ABAD RÍOS ESTRADA,ESP23,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,Postgrado,Postgrado,NUEVO,1-Activo,ENGATIVA,TELECAMPUS,...,0,0,0,0,0,6598000,20000.0,2.400000e+05,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1552,1000133977,JOHAN STIVEN LOZANO GARZON,VT15G,INGENIERIA DE SISTEMAS,Pregrado,Pregrado,NUEVO,1-Activo,BOSA,TELECAMPUS,...,0,0,0,0,0,3653500,53163.0,9.162432e+04,0,0
1553,1000129258,SHIRLEY VANESSA RODRIGUEZ CUELLAR,ESP19,ESPECIALIZACIÓN EN MARKETING DIGITAL,Postgrado,Postgrado,ANTIGUO,1-Activo,BOSA,TELECAMPUS,...,0,0,0,0,0,962062,0.0,2.525155e+05,0,1
1554,1000116616,IVÁN DANIEL RAMOS GAMBA,VTE01,ADMINISTRACIÓN DE EMPRESAS,Pregrado,Pregrado,NUEVO,1-Activo,BOSA,TELECAMPUS,...,0,0,0,0,0,3672400,20000.0,1.586505e+05,0,0
1555,1000115081,JHOAN STEVEN PLATA GUTIERREZ,VTC06,CONTADURIA PUBLICA,Pregrado,Pregrado,NUEVO,1-Activo,BOSA,TELECAMPUS,...,0,0,0,0,0,3672400,0.0,3.442400e+05,0,1


In [ ]:
host_pub = '172.16.1.33'
port_pub = '1433'
database_pub = 'CUN_REPOSITORIO'
url_pub = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_pub, port=port_pub, db=database_pub)
engine_pub = create_engine(url_pub, fast_executemany=True)

table_name_pub = 'HISTORICO_COM_TELEC_2025'
schema_name_pub = 'financiera'
# # Send the data to sql server
df_archivo.to_sql(
     name=table_name_pub,
     schema=schema_name_pub,
     con=engine_pub,
     if_exists='append',
     index=False,
     chunksize = 10000
 )

#  End connection
engine_pub.dispose()